## Environment Setup & Package Imports

This section initializes the required libraries for this notebook. We load standard data manipulation libraries alongside specialized text extraction packages to build our pipeline:

* **`pandas` (`pd`)**: Configures data frames to structure, index, and organize our raw text strings into a clean tabular data matrix.
* **`numpy` (`np`)**: Supports structural missing value representations (`pd.NA`) and numeric vector transformations throughout the pipeline.
* **`os`**: Provides standard operating system interfaces to interact seamlessly with system directories and environments.
* **`re`**: Provides regular expression tools for robust pattern matching and filtering out repetitive PDF artifacts.
* **`pathlib.Path`**: Handles safe, object-oriented, cross-platform system file path configuration.
* **`json`**: Provides standard tools to handle data serialization and state configurations if required.

In [2]:
import re
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# CBO Sentiment & Mapping Pipeline: Synonym Expansion via Word2Vec

## Project Overview
This notebook implements an automated, vocabulary-expanding text-matching pipeline. It bridges the gap between historical raw text blocks extracted from Congressional Budget Office (CBO) reports and structured evaluation datasets containing official federal projection categories. 

By leveraging a pre-trained `Word2Vec` semantic model, the pipeline generates context-aware program synonyms. This allows us to map alternative phrasing, program abbreviations, and semantic neighbors back to their official underlying federal subcategories without manual lookup arrays.

---

## Pipeline Architecture

The pipeline processes data along two parallel tracks—Paragraph Processing versus Semantic Synonym Generation—before intersecting at the regex matching engine:

```text
  [Section 2 & 3: Paragraph Track]            [Section 4 & 5: Semantic Synonym Track]
   ┌────────────────────────┐                ┌───────────────────────────────────┐
   │  Raw Data Extraction   │                │      Word2Vec Model Loading       │
   │ (chunked_paragraphs)   │                │     ("word2vec-google-news")      │
   └───────────┬────────────┘                └─────────────────┬─────────────────┘
               │                                               │
               ▼                                               ▼
   ┌────────────────────────┐                ┌───────────────────────────────────┐
   │   Text Normalization   │                │      Base Vector Extraction       │
   │  & Artifact Filtering  │                │         (label_to_vector)         │
   └───────────┬────────────┘                └─────────────────┬─────────────────┘
               │                                               │
               │                                               ▼
               │                             ┌───────────────────────────────────┐
               │                             │     Context Nudging & Weights     │
               │                             │ (add_context_influence_to_vector) │
               │                             └─────────────────┬─────────────────┘
               │                                               │
               │                                               ▼
               │                             ┌───────────────────────────────────┐
               │                             │       Fetch 1,200 Neighbors       │
               │                             │        (similar_by_vector)        │
               │                             └─────────────────┬─────────────────┘
               │                                               │
               │                                               ▼
               │                             ┌───────────────────────────────────┐
               │                             │     Strict Hygiene Safeguards     │
               │                             │      (keep_candidate_token)       │
               │                             └─────────────────┬─────────────────┘
               │                                               │
               └──────────────────────┬────────────────────────┘
                                      │ (Compiled Synonyms)
                                      ▼
                          ┌────────────────────────┐
                          │Section 6: Phrase Engine│
                          │ (Longest Phrase First) │
                          └───────────┬────────────┘
                                      │
                                      ▼
                          ┌────────────────────────┐
                          │Section 7 & 8: Output   │
                          │  & Execution Summary   │──► [Saved to data_files/]
                          └────────────────────────┘
```

### 1. Semantic Query Nudging
Official CBO subcategories (e.g., *Medicare*, *Social Security*) are converted into dense vector coordinates using the Google News 300-dimension corpus. To align generic vocabulary with budgetary themes, we inject a weighted directional nudge:
$$\vec{V}_{\text{final}} = \alpha \cdot \text{unit}(\vec{V}_{\text{label}}) + \beta \cdot \text{unit}(\vec{V}_{\text{context}})$$
Where $\alpha = 5.0$ (Subcategory Weight) and $\beta = 1.0$ (Context Terms: *spending, outlays, federal, congress, budget*). This forces the model to locate synonyms nested strictly within a federal fiscal framework.

### 2. Guardrails & Hygiene Filtering
Automated semantic expansion carries a high risk of drift (e.g., *Social Security* pulling in *Homeland Security* or *Aviation Safety*). The pipeline applies structural guardrails:
* **Forbidden Token Arrays:** Filters out standalone words (*total, net, discretionary*) and formatting anomalies (*------_Net*).
* **Cross-Program Contamination Checks:** Blocks a synonym if it overlaps with an alternate official subcategory string (e.g., prevents *Medicaid* from registering as a neighbor for *Medicare*).
* **Contextual Bounding:** Drops security tokens associated with defense, intelligence, or border enforcement when parsing social safety net entities.

### 3. Regex Phrase Matcher
Discovered synonyms and official definitions are consolidated into a regular expression matrix. This matrix is sorted **longest-phrase-first** to prioritize high-specificity compound terms over broad ones. Text tokens are isolated using word boundary lookbehinds and lookaheads (`(?<!\w)...(?!\w)`) to prevent partial matching inside unrelated characters.

In [5]:
# ======================================================================
# 1. Pipeline Configuration Settings
# ======================================================================
CHUNK_CSV = "data_files/chunked_paragraphs.csv"
GITHUB_URL = "https://raw.githubusercontent.com/US-CBO/eval-projections/main/output_data/outlay_projection_errors.csv"

OUTPUT_CSV = "data_files/chunked_paragraphs_with_embeddings.csv"
SYNONYM_JSON = "data_files/subcategory_synonyms_word2vec.json"

# Semantic Search & Generation Controls
N_SYNONYMS_PER_LABEL = 20
REGENERATE_SYNONYMS = True
ALWAYS_REWRITE_JSON_AFTER_CLEANING = True

# Vector Space Context Nudging Flags
SYNONYM_CONTEXT_TERMS = ["spending", "outlays", "federal", "congress", "budget"]
SYNONYM_LABEL_WEIGHT = 5.0
SYNONYM_CONTEXT_WEIGHT = 1.0

# Broad/Generic Tokens Denied from Acting as Single Match Criteria
FORBIDDEN_EXACT = {
    "security", "benefits", "insurance", "health", "spending", "outlays", 
    "program", "programs", "mandatory", "discretionary", "total", "net", 
    "interest", "other", "some", "certain", "various", "most", "several", 
    "multiple"
}

FORBIDDEN_SUBSTRINGS = {
    "list", "alternative", "newline", "return", "exactly", "abbreviations"
}

# Cross-Domain Drift Guardrails (Prevents Social Security -> Homeland Security)
SECURITY_CONTEXT_FORBIDDEN = {
    "homeland", "aviation", "transportation", "tsa", "terror", "terrorism", "border"
}


# ======================================================================
# 2. Data Ingestion & Input Verification
# ======================================================================
print("🔄 Initializing CBO Paragraph Enrichment Pipeline...")
df2 = pd.read_csv(CHUNK_CSV).reset_index(drop=True)
df2["row_id"] = np.arange(len(df2), dtype=np.int64)

# Instantiate mandatory framework targeting features
for col in ["component", "category", "subcategory", "match_method"]:
    if col not in df2.columns:
        df2[col] = pd.NA

# Retrieve master structural data mapping array directly from tracking repo
df_errors = pd.read_csv(GITHUB_URL)

mapping = (
    df_errors[["subcategory", "category", "component"]]
    .dropna(subset=["subcategory", "category", "component"])
    .drop_duplicates()
    .drop_duplicates(subset=["subcategory"], keep="first")
)

subcategories = sorted(mapping["subcategory"].unique().tolist())
print(f"✅ Loaded {len(subcategories)} official CBO destination subcategories.")


# ======================================================================
# 3. String Normalization & Structural Cleanup
# ======================================================================
def normalize_text(x):
    """
    Standardizes character typography across parsing windows to protect matching matrix
    against typographical variants (e.g., 'Medi-Cal' == 'Medi_Cal' == 'Medi Cal').
    """
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"[_\-]+", " ", x)     # Convert dash runs and underscores to soft spaces
    x = re.sub(r"[^\w\s]", " ", x)    # Remove punctuation structures
    x = re.sub(r"\s+", " ", x).strip() # Collapse trailing whitespace runs
    return x


def is_formatting_artifact_token(w: str) -> bool:
    """
    Identifies if a string token is a layout artifact or programmatic divider string.
    """
    if not w:
        return True

    w = str(w)
    if re.search(r"[-=]{6,}", w) or re.match(r"^[-=_]{4,}", w):
        return True

    # Check density of letters/digits vs special formatting marks
    letters = len(re.findall(r"[A-Za-z]", w))
    digits = len(re.findall(r"\d", w))
    punct = len(w) - letters - digits

    if letters == 0 and digits == 0:
        return True
    if punct / max(len(w), 1) > 0.55:
        return True
    if re.search(r"_{3,}", w):
        return True

    return False


# ======================================================================
# 4. Semantic Search Vector Processing Engine
# ======================================================================
syn_path = Path(SYNONYM_JSON)

def keep_candidate_token(label: str, w: str) -> bool:
    """
    Evaluates discovered candidate terms against safety arrays to prevent semantic drift.
    Ensures length constraints are met, screens cross-program contamination (e.g., stops 
    Medicaid from leaking into Medicare), and strips out unwanted defense/homeland 
    security terms from social entitlement labels.
    """
    wl = str(w).lower()
    label_l = str(label).lower()

    if is_formatting_artifact_token(w):
        return False
    if any(bad in wl for bad in FORBIDDEN_SUBSTRINGS):
        return False
    if wl in FORBIDDEN_EXACT:
        return False
    if re.fullmatch(r"\d+", str(w)) or len(str(w)) <= 2 or re.search(r"\d", str(w)):
        return False

    # Prevent cross-contamination between program models (e.g., Medicare picking up Medicaid)
    for other in subcategories:
        if other.lower() == label_l:
            continue
        other_norm = normalize_text(other).replace(" ", "_")
        if other_norm and other_norm in wl:
            return False

    # Enforce strict domain safety filtering on entitlement labels
    if label == "Social Security":
        if wl == "security" or any(tok in wl for tok in SECURITY_CONTEXT_FORBIDDEN):
            return False

    return True


def label_to_vector(label: str, model):
    """
    Converts an official text subcategory label into a 1x300 dimensional vector array.
    
    If the subcategory consists of a single word (e.g., "Medicare"), it fetches its 
    exact coordinates from the Word2Vec vocabulary. If it is a multi-word compound 
    label (e.g., "Social Security"), it looks for a pre-joined phrase token 
    ("Social_Security"). If absent, it splits the phrase, fetches the independent 
    300-dimensional arrays for each word, and computes their element-wise arithmetic 
    mean to map the full label to a unified spatial coordinate.

    Parameters:
        label (str): The official subcategory string to find or build.
        model (Word2Vec): Pre-trained semantic news embeddings corpus.

    Returns:
        np.ndarray: A 300-element 1D array of float32 values, or None if unmappable.
    """
    label = str(label).strip()
    phrase_token = label.replace(" ", "_")

    if phrase_token in model:
        return model[phrase_token].astype(np.float32)

    parts = label.split()
    vecs = [model[p].astype(np.float32) for p in parts if p in model]

    if not vecs:
        vecs = [model[p.title()].astype(np.float32) for p in parts if p.title() in model]
    if not vecs:
        return None

    return np.mean(np.vstack(vecs), axis=0).astype(np.float32)


def unit_vec(v):
    """
    Normalizes a 300-dimensional vector's geometric length (magnitude) to exactly 1.0.
    
    This locks the spatial vector to unit space without modifying its semantic directional 
    angle. Normalizing vectors guarantees mathematical equity when blending multiple arrays 
    together later; it prevents heavy, high-frequency background words from completely 
    drowning out rarer, highly-specific category labels.

    Parameters:
        v (np.ndarray): The raw 300-dimensional vector to normalize.

    Returns:
        np.ndarray: The resulting length-normalized unit vector.
    """
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v if n == 0 else v / n


def add_context_influence_to_vector(v, model):
    """
    Blends specific macro-budgetary context vectors onto program vectors via a weighted sum.
    
    This function isolates the 300-dimensional vectors for all domain phrases specified in 
    SYNONYM_CONTEXT_TERMS ("spending", "outlays", "federal", "congress", "budget") and 
    calculates their shared mean to find the center of public finance vocabulary. 
    It then executes a weighted linear blend using a default 5:1 ratio:
    
        V_final = (5.0 * Unit(V_label)) + (1.0 * Unit(V_context))
        
    This alters the target coordinates, forcing subsequent Word2Vec nearest-neighbor searches 
    to pivot away from general consumer meanings (e.g., medical clinical jargon) and shift 
    firmly toward congressional fiscal and budgetary syntax contexts.

    Parameters:
        v (np.ndarray): The initial baseline unit vector for the official label.
        model (Word2Vec): Pre-trained semantic news embeddings corpus.

    Returns:
        np.ndarray: The shifted, context-nudged 300-dimensional unit vector.
    """
    context_vecs = []
    for term in SYNONYM_CONTEXT_TERMS:
        if term in model:
            context_vecs.append(model[term].astype(np.float32))

    if not context_vecs:
        return unit_vec(v).astype(np.float32)

    context_vec = np.mean(np.vstack(context_vecs), axis=0).astype(np.float32)
    combined = (SYNONYM_LABEL_WEIGHT * unit_vec(v)) + (SYNONYM_CONTEXT_WEIGHT * unit_vec(context_vec))
    return unit_vec(combined).astype(np.float32)


# ======================================================================
# 5. Synonym Synthesis Execution Loop
# ======================================================================
if (not syn_path.exists()) or REGENERATE_SYNONYMS:
    print(f"\n🧠 Running semantic expansion via Word2Vec. Injecting spatial anchor terms: {SYNONYM_CONTEXT_TERMS}...")
    import gensim.downloader as api
    w2v_model = api.load("word2vec-google-news-300")

    synonyms = {}
    debug_kept = {}

    for sub in subcategories:
        v = label_to_vector(sub, w2v_model)
        if v is None:
            synonyms[sub] = [sub]
            debug_kept[sub] = 0
            continue

        # Adjust projection maps relative to configuration weights
        v = add_context_influence_to_vector(v, w2v_model)
        nbrs = w2v_model.similar_by_vector(v, topn=1200)

        kept = []
        for w, score in nbrs:
            if w.lower() in {sub.lower(), sub.replace(" ", "_").lower()}:
                continue
            if not keep_candidate_token(sub, w):
                continue
            kept.append(w)
            if len(kept) >= N_SYNONYMS_PER_LABEL:
                break

        synonyms[sub] = [sub] + kept
        debug_kept[sub] = len(kept)

    syn_path.parent.mkdir(parents=True, exist_ok=True)
    syn_path.write_text(json.dumps(synonyms, indent=2), encoding="utf-8")
    print(f"💾 Fresh dictionary file exported to disk at: {SYNONYM_JSON}")

else:
    print(f"\n📂 Loading existing structural lookup arrays from: {SYNONYM_JSON}")
    synonyms = json.loads(syn_path.read_text(encoding="utf-8"))
    cleaned = {}

    for sub, syns in synonyms.items():
        kept = []
        for w in syns:
            if str(w).strip().lower() == str(sub).lower():
                kept.append(sub)
                continue
            if keep_candidate_token(sub, str(w)):
                kept.append(str(w))

        seen = set()
        out = []
        for w in kept:
            k = str(w).lower()
            if k not in seen:
                seen.add(k)
                out.append(w)
        cleaned[sub] = out

    synonyms = cleaned

    if ALWAYS_REWRITE_JSON_AFTER_CLEANING:
        syn_path.parent.mkdir(parents=True, exist_ok=True)
        syn_path.write_text(json.dumps(synonyms, indent=2), encoding="utf-8")
        print(f"🧹 Hygiene routines executed. JSON structure updated at: {SYNONYM_JSON}")


# ======================================================================
# 6. Phrase Matcher Processing Engine Construction
# ======================================================================
phrase_records = []

for sub, syns in synonyms.items():
    for phrase in syns:
        pnorm = normalize_text(phrase)
        if not pnorm or pnorm in FORBIDDEN_EXACT or is_formatting_artifact_token(phrase):
            continue
        phrase_records.append((phrase, sub))

# Sort longest-first to ensure highly explicit compound strings match before short ones
phrase_records = sorted(phrase_records, key=lambda x: len(str(x[0])), reverse=True)
compiled = []

for phrase, sub in phrase_records:
    pnorm = normalize_text(phrase)
    # Enforce strict alphanumeric token boundaries to bypass substring collision errors
    pat = re.compile(r"(?<!\w)" + re.escape(pnorm) + r"(?!\w)", flags=re.IGNORECASE)
    compiled.append((pat, sub, phrase))


def match_by_phrases(text: str):
    """
    Scans normalized input text against compiled regex array to assign matching labels.
    """
    t = normalize_text(text)
    for pat, sub, phrase in compiled:
        if pat.search(t):
            return sub, phrase
    return pd.NA, pd.NA


# ======================================================================
# 7. Document Matching Execution Pipeline
# ======================================================================
print("\n🔎 Deploying Regex Phrase Matrix across extracted text chunks...")

matched_subs = []
matched_phrase = []

for txt in df2["text"].astype(str):
    sub, phrase = match_by_phrases(txt)
    matched_subs.append(sub)
    matched_phrase.append(phrase)

df2["subcategory"] = matched_subs
df2["matched_phrase"] = matched_phrase
df2["match_method"] = np.where(df2["subcategory"].notna(), "direct_phrase_w2v_neighbors", pd.NA)

# Drop structural matching target variables before executing merge mapping
df2 = df2.drop(columns=["category", "component"], errors="ignore").merge(mapping, on="subcategory", how="left")


# ======================================================================
# 8. Clean Structured Output Report Generation
# ======================================================================
Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
df2.to_csv(OUTPUT_CSV, index=False)

# Build a clean scannable dashboard printout
total_rows = len(df2)
matched_count = df2["subcategory"].notna().sum()
unmatched_count = total_rows - matched_count
match_rate = (matched_count / total_rows) * 100

print("\n" + "="*60)
print("                  PIPELINE EXECUTION SUMMARY                 ")
print("="*60)
print(f"  • Total Paragraph Chunks Scanned : {total_rows:,}")
print(f"  • Successful Phrase Matches      : {matched_count:,}")
print(f"  • Unmapped Text Segments         : {unmatched_count:,}")
print(f"  • Target Alignment Success Rate  : {match_rate:.2f}%")
print("-" * 60)
print("🏆 TOP 10 HIGH-FREQUENCY MATCHED SUBCATEGORIES:")
print("-" * 60)

top_subs = df2["subcategory"].value_counts(dropna=False).head(10)
for rank, (sub_name, count) in enumerate(top_subs.items(), 1):
    label_string = str(sub_name) if pd.notna(sub_name) else "[UNMATCHED TEXT CHUNKS]"
    print(f"   {rank:2d}. {label_string:<35} → {count:,} hits")

print("-" * 60)
print(f"💾 Enriched Master File Saved Successfully to: {OUTPUT_CSV}")
print("="*60 + "\nPipeline execution complete.")

🔄 Initializing CBO Paragraph Enrichment Pipeline...
✅ Loaded 10 official CBO destination subcategories.

🧠 Running semantic expansion via Word2Vec. Injecting spatial anchor terms: ['spending', 'outlays', 'federal', 'congress', 'budget']...
💾 Fresh dictionary file exported to disk at: data_files/subcategory_synonyms_word2vec.json

🔎 Deploying Regex Phrase Matrix across extracted text chunks...

                  PIPELINE EXECUTION SUMMARY                 
  • Total Paragraph Chunks Scanned : 34,794
  • Successful Phrase Matches      : 20,321
  • Unmapped Text Segments         : 14,473
  • Target Alignment Success Rate  : 58.40%
------------------------------------------------------------
🏆 TOP 10 HIGH-FREQUENCY MATCHED SUBCATEGORIES:
------------------------------------------------------------
    1. [UNMATCHED TEXT CHUNKS]             → 14,473 hits
    2. Net Interest                        → 5,435 hits
    3. Total Mandatory                     → 3,608 hits
    4. Total               